<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/2_Taller_Regresion_Polinomica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Regresión Polinómica, Subajuste, Sobreajuste y Regularización**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

-Juan Pablo Sanchez Luis
-
-

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Taller_Reg_Pol_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/AUmhqMjhUK.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

30 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

## **Situación**

Una importante firma de inversión inmobiliaria te ha contratado como consultor de ciencia de datos. Su proceso actual de valoración de propiedades es lento y se basa en la intuición de unos pocos expertos. Quieren que desarrolles un modelo de machine learning para predecir el precio de venta de las viviendas (`SalePrice`) de forma más precisa y sistemática.

Te han entregado el dataset "Ames Housing", que contiene una gran cantidad de información sobre viviendas vendidas recientemente. Tu tarea es construir el mejor modelo posible, pero más importante aún, justificar por qué tu modelo es robusto y fiable, explicando cómo has manejado la complejidad y el riesgo de sobreajuste.

### **Ejercicio 1: Carga y Preparación Inicial de los Datos**

Primero, carguemos las librerías necesarias y el dataset. Nos enfocaremos en un subconjunto de las variables numéricas para mantener el taller manejable, pero el principio se aplica a todo el dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

Mejorar visualización de dataframes y gráficos

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

# Configuraciones para una mejor visualización
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

Cargar el dataset

In [3]:
# Usamos el dataset Ames Housing desde su fuente original (requiere internet)
url = 'http://jse.amstat.org/v19n3/decock/AmesHousing.txt'
df = pd.read_csv(url, sep='\t')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.00,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.00,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.00,Unf,0.00,441.00,"1,080.00",GasA,Fa,Y,SBrkr,1656,0,0,1656,1.00,0.00,1,0,3,1,TA,7,Typ,2,Gd,Attchd,"1,960.00",Fin,2.00,528.00,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,CBlock,TA,TA,No,Rec,468.00,LwQ,144.00,270.00,882.00,GasA,TA,Y,SBrkr,896,0,0,896,0.00,0.00,1,0,2,1,TA,5,Typ,0,NaN,Attchd,"1,961.00",Unf,1.00,730.00,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.00,TA,TA,CBlock,TA,TA,No,ALQ,923.00,Unf,0.00,406.00,"1,329.00",GasA,TA,Y,SBrkr,1329,0,0,1329,0.00,0.00,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,"1,958.00",Unf,1.00,312.00,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.00,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.00,Gd,TA,CBlock,TA,TA,No,ALQ,"1,065.00",Unf,0.00,"1,045.00","2,110.00",GasA,Ex,Y,SBrkr,2110,0,0,2110,1.00,0.00,2,1,3,1,Ex,8,Typ,2,TA,Attchd,"1,968.00",Fin,2.00,522.00,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,PConc,Gd,TA,No,GLQ,791.00,Unf,0.00,137.00,928.00,GasA,Gd,Y,SBrkr,928,701,0,1629,0.00,0.00,2,1,3,1,TA,6,Typ,1,TA,Attchd,"1,997.00",Fin,2.00,482.00,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [4]:
# Para este taller, solo usaremos algunas columnas clave para simplificar.
df_sub = df[['Overall Qual', 'Gr Liv Area', 'SalePrice']].copy()
df_sub.columns = ['OverallQual', 'GrLivArea', 'SalePrice'] # Renombrar para facilidad
df_sub.head()

,OverallQual,GrLivArea,SalePrice
0,6,1656,215000
1,5,896,105000
2,6,1329,172000
3,7,2110,244000
4,5,1629,189900


**Explicación de las variables del dataset reducido**

1. SalePrice: Precio de Venta.

Esta es la variable objetivo. Es el precio final por el cual se vendió la propiedad, medido en dólares estadounidenses.

2. OverallQual: Calidad General.

Es una variable ordinal que califica la calidad general del material y el acabado de la casa. Es una de las variables predictoras (X) más importantes.

Escala: Va de 1 a 10.

10: Muy Excelente

9: Excelente

...

2: Pobre

1: Muy Pobre

3. GrLivArea: Área Habitable sobre el Nivel del Suelo.

Es una variable numérica que mide el total de metros cuadrados de área habitable que está por encima del nivel del suelo. No incluye el área del sótano. Es una de las variables predictoras (X) más fuertes, ya que, lógicamente, casas más grandes tienden a ser más caras.

In [5]:
df_sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   OverallQual  2930 non-null   int64
 1   GrLivArea    2930 non-null   int64
 2   SalePrice    2930 non-null   int64
dtypes: int64(3)
memory usage: 68.8 KB


### **Ejercicio 2: Dividir el conjunto de datos**

El método `.info()` muestra que no hay nulos en nuestro subconjunto. ¡Perfecto! Ahora, definamos nuestras variables `X` (predictoras) e `y` (objetivo) y dividamos los datos.

In [6]:
X = df_sub[['OverallQual', 'GrLivArea']]
y = df_sub['SalePrice']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)


### **Ejercicio 3: Modelo con Sobreajuste**

Vamos a crear un modelo muy complejo para ver qué tan mal puede generalizar.

**Crear un Modelo Polinómico de Grado 5**

Usa `Pipeline` para combinar `PolynomialFeatures` (grado 5), `StandardScaler` y `LinearRegression`. Entrénalo con los datos de entrenamiento.

In [8]:
pipeline_poly = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])


In [9]:
pipeline_poly.fit(X_train, y_train)

Pipeline(steps=[('poly_features',
                 PolynomialFeatures(degree=5, include_bias=False)),
                ('scaler', StandardScaler()), ('model', LinearRegression())])

In [10]:
y_train_pred_poly = pipeline_poly.predict(X_train)
y_test_pred_poly  = pipeline_poly.predict(X_test)

rmse_train_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
rmse_test_poly  = np.sqrt(mean_squared_error(y_test,  y_test_pred_poly))

print(f"Modelo Polinómico Grado 5")
print(f"  RMSE Entrenamiento : ${rmse_train_poly:,.2f}")
print(f"  RMSE Prueba        : ${rmse_test_poly:,.2f}")

Modelo Polinómico Grado 5
  RMSE Entrenamiento : $34,152.35
  RMSE Prueba        : $41,101.60


**Pregunta:** ¿Qué indica la diferencia que observas entre el error de entrenamiento y el de prueba? ¿Le recomendarías este modelo a la firma inmobiliaria? ¿Por qué?

El modelo polinómico de grado 5 muestra un RMSE de entrenamiento de 34,152 USD pero sube a 41,101 USD en prueba, una diferencia de casi 7,000 USD. Esto es una señal clara de sobreajuste: el modelo memorizó los datos de entrenamiento incluyendo su ruido, y al enfrentarse a propiedades nuevas su error se dispara casi un 20%.

No recomendaríamos este modelo a la firma inmobiliaria porque en la práctica todas las propiedades a valorar serán datos nuevos, y un error promedio de 41,101 USD cuando prodia obetenerse 35,206 USD con Ridge es inaceptable.

### **Ejercicio 4: Aplicar Regularización**

Ahora, vamos a "curar" el sobreajuste. Usaremos los mismos `PolynomialFeatures` de grado 5, pero cambiaremos el modelo de regresión.

**Implementar Regresión Ridge**

Copia el pipeline anterior, pero reemplaza `LinearRegression` con `Ridge(alpha=10)`. `alpha` es la fuerza de la penalización.

In [11]:
pipeline_ridge = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10))
])

In [12]:
pipeline_ridge.fit(X_train, y_train)


Pipeline(steps=[('poly_features',
                 PolynomialFeatures(degree=5, include_bias=False)),
                ('scaler', StandardScaler()), ('model', Ridge(alpha=10))])

In [13]:
y_train_pred_ridge = pipeline_ridge.predict(X_train)
y_test_pred_ridge  = pipeline_ridge.predict(X_test)

rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
rmse_test_ridge  = np.sqrt(mean_squared_error(y_test,  y_test_pred_ridge))

print(f"Modelo Ridge (alpha=10)")
print(f"  RMSE Entrenamiento : ${rmse_train_ridge:,.2f}")
print(f"  RMSE Prueba        : ${rmse_test_ridge:,.2f}")


Modelo Ridge (alpha=10)
  RMSE Entrenamiento : $35,204.89
  RMSE Prueba        : $35,206.63


**Interpreta los resultados:**
Ridge logra algo clave: la brecha entre entrenamiento y prueba se reduce de 6,949 USD (Polinómico) a solo 2 USD (35,204 vs 35,206). Aunque el error de entrenamiento sube ligeramente respecto al modelo polinómico puro, el de prueba baja 5,895 USD, que es lo que realmente importa. Ridge penaliza coeficientes grandes y evita que el modelo se aferre al ruido de los datos de entrenamiento, generalizando de forma muy estable.

**Implementar Regresión Lasso**

Haz lo mismo, pero ahora con `Lasso(alpha=500, max_iter=10000)`. Lasso necesita un `alpha` más grande (porque la penalización L1 es diferente) y a veces más `max_iter` para converger.

In [14]:
pipeline_lasso = Pipeline([
    ('poly_features', PolynomialFeatures(degree=5, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=500, max_iter=10000))
])


In [15]:
pipeline_lasso.fit(X_train, y_train)

Pipeline(steps=[('poly_features',
                 PolynomialFeatures(degree=5, include_bias=False)),
                ('scaler', StandardScaler()),
                ('model', Lasso(alpha=500, max_iter=10000))])

In [16]:
y_train_pred_lasso = pipeline_lasso.predict(X_train)
y_test_pred_lasso  = pipeline_lasso.predict(X_test)

rmse_train_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
rmse_test_lasso  = np.sqrt(mean_squared_error(y_test,  y_test_pred_lasso))

print(f"Modelo Lasso (alpha=500)")
print(f"  RMSE Entrenamiento : ${rmse_train_lasso:,.2f}")
print(f"  RMSE Prueba        : ${rmse_test_lasso:,.2f}")

Modelo Lasso (alpha=500)
  RMSE Entrenamiento : $35,352.92
  RMSE Prueba        : $35,447.71


**Interpreta los resultados:**
Lasso también logra controlar el sobreajuste de forma efectiva. La brecha entre el RMSE de entrenamiento (35,352) y el de prueba (35,447) es de apenas 95 USD, lo que indica que el modelo generaliza muy bien a datos nuevos, similar a Ridge.
Sin embargo, su verdadero valor no está solo en el error sino en lo que hace internamente: de las 20 características polinómicas disponibles, eliminó 14 (70%), quedándose únicamente con las 6 que realmente explican el precio. Esto lo convierte en un modelo igual de preciso que Ridge pero más simple e interpretable, ya que trabaja solo con las variables que tienen poder predictivo real y descarta el ruido matemático automáticamente.

**Selección de Variables con Lasso**

Una de las grandes ventajas de Lasso es que puede eliminar variables. Vamos a ver cuántas de las características polinómicas que creamos (ej. `OverallQual^2`, `GrLivArea^5`, `OverallQual * GrLivArea^4`, etc.) fueron eliminadas.

In [17]:
lasso_coefs = pipeline_lasso.named_steps['model'].coef_

In [18]:
feature_names = pipeline_lasso.named_steps['poly_features'].get_feature_names_out(X.columns)

In [19]:
coefs_cero = np.sum(lasso_coefs == 0)


In [20]:
total_features     = len(feature_names)
features_eliminadas = coefs_cero
features_conservadas = total_features - features_eliminadas
pct_eliminadas     = (features_eliminadas / total_features) * 100

nombres_conservadas = feature_names[lasso_coefs != 0]

print(f"Total de características polinómicas generadas : {total_features}")
print(f"Características eliminadas por Lasso (coef=0)  : {features_eliminadas}")
print(f"Características conservadas                    : {features_conservadas}")
print(f"Porcentaje de variables eliminadas             : {pct_eliminadas:.1f}%")
print(f"\nCaracterísticas conservadas:")
for nombre in nombres_conservadas:
    print(f"  - {nombre}")

Total de características polinómicas generadas : 20
Características eliminadas por Lasso (coef=0)  : 14
Características conservadas                    : 6
Porcentaje de variables eliminadas             : 70.0%

Características conservadas:
  - OverallQual
  - OverallQual GrLivArea
  - OverallQual^2 GrLivArea
  - OverallQual^3 GrLivArea
  - OverallQual^5
  - GrLivArea^5


**Interpreta los resultados:**

De las 20 características polinómicas generadas a partir de OverallQual y GrLivArea, Lasso eliminó 14, es decir el 70%, conservando únicamente 6. Esto confirma que la gran mayoría de combinaciones de alto grado eran ruido matemático sin valor predictivo real.

Las 6 variables conservadas tienen una lógica económica clara:

-OverallQual: la calidad general por sí sola es el predictor más directo del precio. Una casa mejor construida vale más, sin importar su tamaño.

-OverallQual × GrLivArea: la interacción entre calidad y tamaño es clave. Una casa grande pero de mala calidad no vale lo mismo que una grande y bien construida.

-OverallQual² × GrLivArea y OverallQual³ × GrLivArea: capturan que el efecto de la calidad sobre el precio no es lineal. A medida que la calidad sube, el precio crece de forma acelerada, especialmente en casas grandes.

-OverallQual⁵: refleja que en los niveles más altos de calidad (9-10), el precio se dispara de forma exponencial, un fenómeno típico del mercado inmobiliario de lujo.

-GrLivArea⁵: el área habitable también tiene un efecto no lineal en el extremo superior: las propiedades muy grandes tienen un premium de precio desproporcionado.

En resumen, Lasso descartó automáticamente el 70% de las variables y se quedó solo con las que capturan la relación real entre calidad, tamaño y precio.

### **Ejercicio 5: Conclusión y Recomendación para el Cliente**

**Resumir los Resultados**

Crea un `DataFrame` de pandas que compare el RMSE de entrenamiento y prueba de los tres modelos (Polinómico, Ridge, Lasso) y ordena los modelos según el RMSE de prueba de menor a mayor.

In [21]:
resultados = pd.DataFrame({
    'Modelo': ['Polinómico Grado 5', 'Ridge (alpha=10)', 'Lasso (alpha=500)'],
    'RMSE Entrenamiento': [rmse_train_poly, rmse_train_ridge, rmse_train_lasso],
    'RMSE Prueba':        [rmse_test_poly,  rmse_test_ridge,  rmse_test_lasso]
})

resultados = resultados.sort_values('RMSE Prueba').reset_index(drop=True)
resultados


,Modelo,RMSE Entrenamiento,RMSE Prueba
0,Ridge (alpha=10),"35,204.89","35,206.63"
1,Lasso (alpha=500),"35,352.92","35,447.71"
2,Polinómico Grado 5,"34,152.35","41,101.60"


**Pregunta Final:**

Basado en tu análisis, ¿qué modelo le recomendarías a la firma inmobiliaria? Tu respuesta debe incluir:

1.  Una recomendación clara del mejor modelo.
2.  Una explicación en términos sencillos (para un gerente, no para un científico de datos) de por qué el modelo polinómico simple no era una buena idea, usando el concepto de "memorizar vs. generalizar".
3.  Una descripción de la ventaja principal del modelo Lasso en cuanto a la interpretabilidad y la selección de las características más importantes.

Recomendación:Recomendamos Ridge, ya que obtiene el menor RMSE de prueba (35,206 USD) de los tres modelos, con una estabilidad casi perfecta entre entrenamiento y prueba. Si la interpretabilidad es prioritaria para la firma, Lasso es una alternativa válida con solo 241 USD más de error.

Para el gerente:El modelo polinómico actúa como un tasador que memoriza de memoria cada venta histórica, incluyendo los casos atípicos y errores del pasado. Cuando le presentas una propiedad nueva, falla porque no aprendió la lógica del mercado sino los datos de memoria. Eso se ve en los números: con propiedades conocidas se equivoca 34,152 USD, pero con propiedades nuevas el error sube a 41,101 USD. Ridge y Lasso, en cambio, aprendieron los patrones reales del mercado y mantienen un error consistente alrededor de 35,000 USD tanto en datos conocidos como nuevos.

Ventaja de Lasso en interpretabilidad:En lugar de depender de 20 combinaciones matemáticas, Lasso identificó automáticamente que solo 6 variables realmente determinan el precio. Esto le permite a la firma comunicarle a sus clientes e inversionistas de forma clara y directa qué factores de una propiedad mueven su valor: principalmente la calidad de construcción y su interacción con el tamaño habitable, descartando el resto como ruido estadístico.